In [ ]:
# Install to the same Python the kernel uses
import sys
!{sys.executable} -m pip install altair vega_datasets pandas --quiet

In [ ]:
import altair as alt
import pandas as pd
from vega_datasets import data as vega_data

print("Altair version:", alt.__version__)

In [ ]:
# Load the data and count organizations per state
df = pd.read_csv("SAFFRON TIES  - CLEAN.csv")
df['STATE'] = df['STATE'].str.strip()

# Count orgs per state
state_counts = df.groupby('STATE').size().reset_index(name='count')

# Map full state names to 2-letter abbreviations (needed for the map)
state_abbrev = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR',
    'CALIFORNIA': 'CA', 'COLORADO': 'CO', 'CONNECTICUT': 'CT', 'DELAWARE': 'DE',
    'FLORIDA': 'FL', 'GEORGIA': 'GA', 'HAWAII': 'HI', 'IDAHO': 'ID',
    'ILLINOIS': 'IL', 'INDIANA': 'IN', 'IOWA': 'IA', 'KANSAS': 'KS',
    'KENTUCKY': 'KY', 'LOUISIANA': 'LA', 'MAINE': 'ME', 'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN', 'MISSISSIPPI': 'MS',
    'MISSOURI': 'MO', 'MONTANA': 'MT', 'NEBRASKA': 'NE', 'NEVADA': 'NV',
    'NEW HAMPSHIRE': 'NH', 'NEW JERSEY': 'NJ', 'NEW MEXICO': 'NM', 'NEW YORK': 'NY',
    'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH', 'OKLAHOMA': 'OK',
    'OREGON': 'OR', 'PENNSYLVANIA': 'PA', 'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD', 'TENNESSEE': 'TN', 'TEXAS': 'TX', 'UTAH': 'UT',
    'VERMONT': 'VT', 'VIRGINIA': 'VA', 'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV',
    'WISCONSIN': 'WI', 'WYOMING': 'WY', 'DISTRICT OF COLUMBIA': 'DC'
}

state_counts['id'] = state_counts['STATE'].map(state_abbrev)
print(state_counts)

In [ ]:
# US states GeoJSON (from vega_datasets, includes numeric FIPS id)
states_url = "https://cdn.jsdelivr.net/npm/vega-datasets@2/data/us-10m.json"

# Map state abbreviations to FIPS numeric IDs (required to join with the map topojson)
fips = {
    'AL':1,'AK':2,'AZ':4,'AR':5,'CA':6,'CO':8,'CT':9,'DE':10,'DC':11,
    'FL':12,'GA':13,'HI':15,'ID':16,'IL':17,'IN':18,'IA':19,'KS':20,
    'KY':21,'LA':22,'ME':23,'MD':24,'MA':25,'MI':26,'MN':27,'MS':28,
    'MO':29,'MT':30,'NE':31,'NV':32,'NH':33,'NJ':34,'NM':35,'NY':36,
    'NC':37,'ND':38,'OH':39,'OK':40,'OR':41,'PA':42,'RI':44,'SC':45,
    'SD':46,'TN':47,'TX':48,'UT':49,'VT':50,'VA':51,'WA':53,'WV':54,
    'WI':55,'WY':56
}

state_counts['fips'] = state_counts['id'].map(fips)

# Build the choropleth map
map_data = alt.topo_feature(states_url, 'states')

choropleth = alt.Chart(map_data).mark_geoshape().transform_lookup(
    lookup='id',
    from_=alt.LookupData(state_counts, 'fips', ['count', 'STATE'])
).encode(
    color=alt.Color(
        'count:Q',
        scale=alt.Scale(scheme='orangered'),
        legend=alt.Legend(title='Number of Organizations')
    ),
    tooltip=['STATE:N', 'count:Q']
).project(
    type='albersUsa'
).properties(
    width=700,
    height=450,
    title='Saffron Ties Organizations by State'
)

choropleth